# Change Detection of Vegetation Around the Port of Savannah Using Sentinel-2 Imagery

This notebook preprocesses Sentinel-2 imagery from 2016, 2020, and 2024 to quantify vegetation change surrounding the Port of Savannah.

Major workflow steps include:

- cloud and invalid-pixel masking
- clipping imagery to the study area
- conversion to GeoTIFF
- NDVI calculation
- ΔNDVI change detection

1. Project Configuration
2. Import Libraries
3. Helper Functions
4. Preprocess Sentinel-2 Imagery
5. Calculate NDVI
6. Perform Change Detection
7. Generate Figures
8. Results
9. Conclusions

 ## 1. Project Configuration

In [34]:
from pathlib import Path

# Root project directory
PROJECT_DIR = Path(r"..")

# Input imagery
RAW_DIR = PROJECT_DIR / "data" / "raw"

# Study area boundary
AOI = PROJECT_DIR / "data" / "study_area" / "study_area.geojson"
print(AOI)
# Output directories
OUTPUT_DIR = PROJECT_DIR / "outputs"

MASK_DIR = OUTPUT_DIR / "masked"

CLIP_DIR = OUTPUT_DIR / "clipped"

NDVI_DIR = OUTPUT_DIR / "ndvi"

FIGURE_DIR = OUTPUT_DIR / "figures"

# Years included in the analysis
IMAGE_FOLDERS = {
    2016: "S2A_MSIL2A_2016",
    2020: "S2A_MSIL2A_2020",
    2024: "S2B_MSIL2A_2024"
}   

# Sentinel-2 band names
BLUE_BAND = "B02_20m"

GREEN_BAND = "B03_20m"

RED_BAND = "B04_20m"

NIR_BAND = "B8A_20m"

SCL_BAND = "SCL_20m"

..\data\study_area\study_area.geojson


In [35]:
# Display the configuration settings

print("Project Configuration")
print("---------------------")
print(f"Project Directory : {PROJECT_DIR}")
print(f"Study Area        : {AOI}")
print(f"Image Folders     : {list(IMAGE_FOLDERS.keys())}")
print(f"Output Directory  : {OUTPUT_DIR}")

Project Configuration
---------------------
Project Directory : ..
Study Area        : ..\data\study_area\study_area.geojson
Image Folders     : [2016, 2020, 2024]
Output Directory  : ..\outputs


## 2. Import Libraries

This notebook uses Rasterio for raster processing, GeoPandas for vector data, NumPy for numerical calculations, and Matplotlib for generating publication-quality figures.

In [36]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import rasterio
from rasterio.mask import mask

import geopandas as gpd

# Verify environment versions
print(f"rasterio version: {rasterio.__version__}")
print(f"NumPy version: {np.__version__}")

rasterio version: 1.3.6
NumPy version: 1.26.4


## 3. Helper Functions

In [37]:
def find_band(image_folder, band_name):
    """
    Locate a Sentinel-2 band within a SAFE directory.

    Parameters
    ----------
    image_folder : Path
        Root SAFE directory.

    band_name : str
        Sentinel-2 band identifier
        (e.g., B04_20m).

    Returns
    -------
    Path
        Full path to the requested band.
    """

    for path in image_folder.rglob("*.jp2"):
        if band_name.lower() in path.name.lower():
            return path

    raise FileNotFoundError(f"{band_name} not found.")

In [38]:
def clip_raster(input_raster, output_raster, geometry):
    """
    Clip a raster using the study area boundary.
    """

    with rasterio.open(input_raster) as src:

        clipped_image, clipped_transform = mask(
            src,
            geometry,
            crop=True
        )

        metadata = src.meta.copy()

        metadata.update(
            {
                "driver": "GTiff",
                "height": clipped_image.shape[1],
                "width": clipped_image.shape[2],
                "transform": clipped_transform
            }
        )

        with rasterio.open(output_raster, "w", **metadata) as dest:

            dest.write(clipped_image)

In [39]:
def print_step(message):
    """
    Print a formatted processing message.
    """

    print(f"\n{'=' * 60}")
    print(message)
    print(f"{'=' * 60}")

In [40]:
def verify_file(filepath):
    """
    Verify that a required file exists.
    """

    if not filepath.exists():
        raise FileNotFoundError(filepath)

    return filepath

In [41]:
def get_year_directory(parent_dir, year):
    """
    Create and return an output directory
    for a specific acquisition year.
    """

    folder = parent_dir / str(year)
    folder.mkdir(parents=True, exist_ok=True)

    return folder

In [42]:
def save_raster(array, reference_raster, output_path, nodata=np.nan):
    """
    Save a NumPy array as a GeoTIFF using another raster as
    the spatial reference.
    """

    with rasterio.open(reference_raster) as src:

        profile = src.profile.copy()

        profile.update(
            driver="GTiff",
            dtype=rasterio.float32,
            nodata=np.nan,
            compress="lzw"
        )

        with rasterio.open(output_path, "w", **profile) as dst:

            dst.write(array.astype(rasterio.float32), 1)

In [43]:
def create_rgb_figure(year):
    """
    Create RGB composite figure for a given year 
    and save it to the specified output path.
    """
    print_step(f"Creating RGB Composite ({year})")
    
    with rasterio.open(CLIP_DIR / str(year) / f"{BLUE_BAND}.tif") as src:
        blue = src.read(1)

    with rasterio.open(CLIP_DIR / str(year) / f"{GREEN_BAND}.tif") as src:
        green = src.read(1)

    with rasterio.open(CLIP_DIR / str(year) / f"{RED_BAND}.tif") as src:
        red = src.read(1)

    rgb = np.dstack((red, green, blue))

    rgb = rgb.astype(float)
    rgb /= np.percentile(rgb, 98)

    rgb = np.clip(rgb, 0, 1)

    plt.figure(figsize=(8,8))
    plt.imshow(rgb)
    plt.title(f"RGB Composite ({year})")
    plt.axis("off")

    plt.savefig(
        FIGURE_DIR / f"rgb_{year}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

## 4. Preprocess Sentinel-2 Imagery

Sentinel-2 imagery is preprocessed before analysis to remove unwanted pixels and standardize the datasets across all acquisition years.

The preprocessing workflow consists of:

1. Locating the required Sentinel-2 spectral bands.
2. Applying the Scene Classification Layer (SCL) mask to remove clouds, shadows, and other invalid pixels.
3. Clipping each masked raster to the study area.

The resulting rasters provide consistent inputs for NDVI calculation and change detection.

In [ ]:
print_step("Preprocessing Sentinel-2 Imagery")

study_area = gpd.read_file(AOI)

# Determine the CRS of the raster data using the red band from 2016
with rasterio.open(find_band(RAW_DIR / IMAGE_FOLDERS[2016], RED_BAND)) as src:
    raster_crs = src.crs

# Reproject the study area to the raster CRS
study_area = study_area.to_crs(raster_crs)

geometry = study_area.geometry.values

bands = [
    BLUE_BAND,
    GREEN_BAND,
    RED_BAND,
    NIR_BAND,
]

for year, folder in IMAGE_FOLDERS.items():

    print(f"\nProcessing {year}")

    year_folder = RAW_DIR / folder

    scl_path = find_band(year_folder, SCL_BAND)

    with rasterio.open(scl_path) as scl_src:

        scl = scl_src.read(1)

    masked_folder = get_year_directory(MASK_DIR, year)

    clipped_folder = get_year_directory(CLIP_DIR, year)

    for band in bands:

        band_path = find_band(year_folder, band)

        with rasterio.open(band_path) as src:

            image = src.read(1)

            valid = np.isin(scl, [4,5,6,7])

            masked = np.where(valid, image, np.nan)

            profile = src.profile.copy()

            profile.update(
                driver="GTiff",
                dtype=rasterio.float32,
                nodata=np.nan,
                compress="lzw"
            )

            temp_path = masked_folder / f"{band}.tif"

            with rasterio.open(temp_path, "w", **profile) as dst:

                dst.write(masked.astype(rasterio.float32),1)

        clip_raster(
            temp_path,
            clipped_folder / f"{band}.tif",
            geometry
        )

    print(f"Finished {year}")


Preprocessing Sentinel-2 Imagery

Processing 2016


Study Area CRS:
PROJCS["WGS 84 / UTM zone 17N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-81],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32617"]]

Raster CRS:
EPSG:32617

Raster Bounds:
BoundingBox(left=399960.0, bottom=3490200.0, right=509760.0, top=3600000.0)

Study Area Bounds:
[ 475612.66752852 3535588.16486675  513072.12771299 3559279.29617473]
Study Area CRS:
PROJCS["WGS 84 / UTM zone 17N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM[

FileNotFoundError: SCL_20m not found.

In [ ]:
# Confirm each raster was produced

print_step("Preprocessing Complete")

for year, folder_name in IMAGE_FOLDERS.items():

    folder = CLIP_DIR / str(year)

    print(f"{year}")

    for raster in folder.glob("*.tif"):

        print(f"   {raster.name}")

## 5. Calculate NDVI and Summary Statistics

The Normalized Difference Vegetation Index (NDVI) is calculated for each acquisition year using the red and near-infrared bands.

\[
NDVI = \frac{NIR - Red}{NIR + Red}
\]

Healthy vegetation strongly reflects near-infrared energy while absorbing red light, resulting in higher NDVI values.

The resulting rasters provide the basis for multi-temporal vegetation change analysis.

In [ ]:
print_step("Calculating NDVI")

for year in IMAGE_FOLDERS.keys():

    print(f"\n{year}")

    folder = CLIP_DIR / str(year)

    with rasterio.open(folder / f"{RED_BAND}.tif") as src:

        red = src.read(1).astype(np.float32)

        reference = folder / f"{RED_BAND}.tif"

    with rasterio.open(folder / f"{NIR_BAND}.tif") as src:

        nir = src.read(1).astype(np.float32)

    np.seterr(divide="ignore",
              invalid="ignore")

    ndvi = (nir-red)/(nir+red)

    output = NDVI_DIR / f"NDVI_{year}.tif"

    save_raster(
        ndvi,
        reference,
        output
    )

    print(f"Saved {output.name}")

In [ ]:
print_step("NDVI Summary Statistics")

for year in IMAGE_FOLDERS.keys():

    with rasterio.open(
        NDVI_DIR / f"NDVI_{year}.tif"
    ) as src:

        ndvi = src.read(1)

    valid = ndvi[np.isfinite(ndvi)]

    print(f"\n{year}")

    print(f"Mean NDVI : {valid.mean():.3f}")
    print(f"Minimum   : {valid.min():.3f}")
    print(f"Maximum   : {valid.max():.3f}")
    print(f"Std Dev   : {valid.std():.3f}")

# 6. Multi-Temporal Change Detection

Vegetation change is quantified by subtracting the NDVI raster from the earliest acquisition year (2016) from the most recent acquisition year (2024).

\[
\Delta NDVI = NDVI_{2024} - NDVI_{2016}
\]

Positive values indicate increased vegetation density, while negative values indicate vegetation loss.

This raster provides a spatial representation of vegetation change across the study area.

In [ ]:
print_step("Calculating Change Detection")

with rasterio.open(
    NDVI_DIR / "NDVI_2016.tif"
) as src:

    ndvi2016 = src.read(1)

    reference = NDVI_DIR / "NDVI_2016.tif"

with rasterio.open(
    NDVI_DIR / "NDVI_2024.tif"
) as src:

    ndvi2024 = src.read(1)

delta = ndvi2024 - ndvi2016

output = NDVI_DIR / "NDVI_delta_2016_2024.tif"

save_raster(
    delta,
    reference,
    output
)

print(f"Saved {output.name}")

In [ ]:
print_step("Vegetation Change Statistics")

valid = delta[np.isfinite(delta)]

gain = np.sum(valid > 0)
loss = np.sum(valid < 0)
stable = np.sum(valid == 0)

total = valid.size

print(f"Mean ΔNDVI : {valid.mean():.3f}")
print(f"Minimum    : {valid.min():.3f}")
print(f"Maximum    : {valid.max():.3f}")
print(f"Std Dev    : {valid.std():.3f}")

print()

print(f"Vegetation Gain : {gain/total*100:.1f}%")
print(f"Vegetation Loss : {loss/total*100:.1f}%")
print(f"No Change       : {stable/total*100:.1f}%")

# 7. Generate Figures
1. RGB Composite 2016
2. RGB Composite 2024
3. NDVI (2024)
4. ΔNDVI
5. ΔNDVI Histogram

In [ ]:
print_step("Creating NDVI Figures")

create_rgb_figure(2016)
create_rgb_figure(2024)

In [ ]:
print_step("Creating Change Detection Figure")

with rasterio.open(
    NDVI_DIR / "NDVI_delta_2016_2024.tif"
) as src:

    delta = src.read(1)

plt.figure(figsize=(9,9))

plt.imshow(
    delta,
    cmap="RdYlGn",
    vmin=-0.5,
    vmax=0.5
)

plt.colorbar(label="ΔNDVI")

plt.title("Vegetation Change (2016–2024)")

plt.axis("off")

plt.savefig(
    FIGURE_DIR / "delta_ndvi.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
print_step("Creating ΔNDVI Histogram")

valid = delta[np.isfinite(delta)]

plt.figure(figsize=(8,5))

plt.hist(valid, bins=50)

plt.xlabel("ΔNDVI")

plt.ylabel("Pixel Count")

plt.title("Distribution of Vegetation Change")

plt.savefig(
    FIGURE_DIR / "delta_histogram.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()